# House Price Prediction - Exploratory Data Analysis & Modeling

This notebook covers the complete machine learning pipeline for predicting house prices using synthetic data. The steps include:
1. **Data loading and inspection**: Understanding the features.
2. **Exploratory Data Analysis (EDA)**: Finding correlations and feature distributions.
3. **Model Selection**: Comparing Multiple Linear Regression and Random Forest Regressor.
4. **Hyperparameter Tuning**: Finding the best parameters using Grid Search.
5. **Validation & Evaluation**: Assessing model generalization using cross-validation and metrics (MAE, RMSE, R²).

## 1. Imports and Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import joblib

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load the Dataset

In [ ]:
# Load the generated dataset
dataset_path = 'dataset.csv'
if not os.path.exists(dataset_path):
    # Fallback if run from parent folder
    dataset_path = os.path.join('house-price-prediction', 'dataset.csv')

df = pd.read_csv(dataset_path)
print(f"Dataset shape: {df.shape}")
df.head()

### Descriptive Statistics

In [ ]:
df.describe()

## 3. Exploratory Data Analysis

### Feature Distribution
Let's look at how the target variable (`price`) and key features are distributed.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price distribution
sns.histplot(df['price'], kde=True, ax=axes[0,0], color='teal')
axes[0,0].set_title('House Price Distribution')
axes[0,0].set_xlabel('Price ($)')

# Sqft Living distribution
sns.histplot(df['sqft_living'], kde=True, ax=axes[0,1], color='coral')
axes[0,1].set_title('Living Space Distribution')
axes[0,1].set_xlabel('Sqft')

# Distance distribution
sns.histplot(df['distance_to_center_km'], kde=True, ax=axes[1,0], color='purple')
axes[1,0].set_title('Distance to City Center Distribution')
axes[1,0].set_xlabel('Distance (km)')

# Age distribution
sns.histplot(df['age_years'], kde=True, ax=axes[1,1], color='gold')
axes[1,1].set_title('House Age Distribution')
axes[1,1].set_xlabel('Age (Years)')

plt.tight_layout()
plt.show()

### Correlations

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

## 4. Modeling

In [ ]:
# Split features and target
X = df.drop(columns=['price'])
y = df['price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

### 4.1 Linear Regression (Baseline)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Cross validation
lr_cv = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='r2')
print(f"Linear Regression CV R2 Score: {np.mean(lr_cv):.4f}")

### 4.2 Random Forest Regressor & Hyperparameter Tuning

In [ ]:
rf_base = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

rf_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Random Forest CV R2 Score: {grid_search.best_score_:.4f}")

## 5. Evaluation

In [ ]:
models = {
    'Linear Regression': lr_model,
    'Random Forest': rf_model
}

for name, model in models.items():
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = mean_absolute_percentage_error(y_test, preds) * 100
    r2 = r2_score(y_test, preds)
    
    print(f"\n=== {name} Test Set Evaluation ===")
    print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
    print(f"R2 Score: {r2:.4f}")

### Actual vs Predicted Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (name, model) in enumerate(models.items()):
    preds = model.predict(X_test)
    axes[i].scatter(y_test, preds, alpha=0.5, color='teal' if i == 0 else 'coral')
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
    axes[i].set_title(f"{name}: Actual vs Predicted")
    axes[i].set_xlabel('Actual Price ($)')
    axes[i].set_ylabel('Predicted Price ($)')

plt.tight_layout()
plt.show()

### Residual Analysis

In [ ]:
rf_preds = rf_model.predict(X_test)
residuals = y_test - rf_preds

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='purple', bins=30)
plt.axvline(0, color='red', linestyle='--')
plt.title('Residuals Distribution (Random Forest)')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.show()

### Feature Importances

In [ ]:
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()